In [35]:
import pprint
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

In [36]:
#load data set
iris = load_iris()

#convert to dataframe
data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
data['species'] = iris.target

data['species'] = data['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

In [37]:
data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [46]:
def calcute_entropy(target_feature):
    # Get the unique class labels and their frequency counts
    classes, counts = np.unique(target_feature, return_counts=True)
    
    # Compute entropy using the formula:
    #   Entropy = - Σ (p_i * log2(p_i))
    # where p_i is the probability of each class
    entropy = np.sum([
        (-counts[i] / np.sum(counts)) * np.log2(counts[i] / np.sum(counts))
        for i in range(len(classes))
    ])
    
    # Return the entropy value for the given target distribution
    return entropy


In [39]:
def best_id3_cutpoint(col, target):
    """
    Find the optimal binary split point for a numerical feature
    according to the ID3 algorithm using Information Gain.
    
    Steps:
    1) Sort unique values of the feature.
    2) Generate candidate cut-points as midpoints between consecutive values.
    3) For each cut:
       - Split target into left (<= cut) and right (> cut)
       - Compute weighted entropy after the split
       - Compute Information Gain
    4) Return the cut with the maximum Information Gain.
    """

    # Sort unique values to create candidate midpoints
    sorted_vals = np.sort(col.unique())
    
    # Midpoints between consecutive values serve as candidate splits
    candidates = [
        (sorted_vals[i] + sorted_vals[i+1]) / 2 
        for i in range(len(sorted_vals)-1)
    ]

    best_gain = -1
    best_cut = None

    # Entropy before any split
    base_entropy = calcute_entropy(target)

    # Evaluate each candidate cut
    for c in candidates:
        # Partition data based on the candidate cut
        left = target[col <= c]
        right = target[col > c]

        # Weighted average entropy after the split
        weighted_entropy = (
            (len(left) / len(col)) * calcute_entropy(left) +
            (len(right) / len(col)) * calcute_entropy(right)
        )

        # Information Gain = Entropy_before - Entropy_after
        gain = base_entropy - weighted_entropy

        # Keep the best split
        if gain > best_gain:
            best_gain = gain
            best_cut = c

    return best_cut, best_gain




In [40]:
def best_k_cutpoints(col, target, k=3):
    """
    Find k best cut-points using a greedy ID3-style algorithm.
    Repeatedly find the best binary split on the largest segment.
    """
    # initial interval = full data
    intervals = [(col, target)]
    cuts = []

    for _ in range(k):
        # find interval with largest entropy
        idx = np.argmax([calcute_entropy(t) for _, t in intervals])
        col_seg, target_seg = intervals.pop(idx)

        # find best split on this segment
        cut, gain = best_id3_cutpoint(col_seg, target_seg)

        cuts.append(cut)

        # split segment into left & right parts
        left_mask = col_seg <= cut
        right_mask = col_seg > cut

        intervals.append((col_seg[left_mask], target_seg[left_mask]))
        intervals.append((col_seg[right_mask], target_seg[right_mask]))

    cuts_sorted = sorted(cuts)
    return cuts_sorted

In [44]:
def info_gain(data, split_attribute_name, target_name):
    # Compute the entropy of the target variable before splitting
    total_entropy = calcute_entropy(target_name)
    
    # Get unique values of the feature and their counts
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    
    # Compute the weighted entropy after splitting on the feature
    weighted_entropy = np.sum([
        (counts[i] / np.sum(counts)) * 
        calcute_entropy(
            data.where(data[split_attribute_name] == vals[i]).dropna()[target_name]
        )
        for i in range(len(vals))
    ])
    
    # Information Gain = entropy before split - entropy after split
    info_gain = total_entropy - weighted_entropy
    
    # Return calculated information gain for this feature
    return info_gain


In [45]:
def id3(data, original_data, features, target_attribute_name, parent_node_class=None):
    # Case 1: If all target values are identical → return that class (pure node)
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]
    
    # Case 2: If dataset becomes empty → return the majority class from the original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[
            np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])
        ]
    
    # Case 3: If no features left to split on → return the parent node's majority class
    elif len(features) == 0:
        return parent_node_class
    
    else:
        # Compute parent node majority class for fallback
        parent_node_class = np.unique(data[target_attribute_name])[
            np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])
        ]
        
        # Compute information gain for each feature
        item_values = [info_gain(data, feature, target_attribute_name) for feature in features]
        
        # Select the feature with the highest information gain
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]
        
        # Initialize tree with the best feature as root
        tree = {best_feature: {}}
        
        # Remove the selected feature from the list
        features = [f for f in features if f != best_feature]
        
        # Create branches for each unique value of the selected feature
        for value in np.unique(data[best_feature]):
            # Filter the dataset for the current value
            sub_data = data.where(data[best_feature] == value).dropna()
            
            # Recursively build a subtree
            subtree = id3(sub_data, data, features, target_attribute_name, parent_node_class)
            
            # Assign subtree to the corresponding branch
            tree[best_feature][value] = subtree
        
        # Return the constructed decision tree
        return tree


In [41]:
data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [42]:
featutes=['sepal length (cm)','sepal width (cm)','petal length (cm)','petal width (cm)']
categories=['sepal_length_category','sepal_width_category','petal_length_category','petal_width_category']

for i in range(len(featutes)):
    cutpoints= best_k_cutpoints(data[featutes[i]], data['species'], k=3)
    finite_edges = sorted([x for x in cutpoints if np.isfinite(x)])

    # Add -inf and +inf if necessary
    processed_bins = []
    if not any(x == -np.inf for x in cutpoints):
        processed_bins.append(-np.inf)
    processed_bins.extend(finite_edges)
    if not any(x == np.inf for x in cutpoints):
        processed_bins.append(np.inf)

    # Ensure unique increasing
    unique_increasing_bins = []
    if processed_bins:
        unique_increasing_bins.append(processed_bins[0])
        for j in range(1, len(processed_bins)):   # <‑‑ اینجا i را به j تغییر دادم
            if processed_bins[j] > unique_increasing_bins[-1]:
                unique_increasing_bins.append(processed_bins[j])

    cut_processed = unique_increasing_bins   

    labels=['Very Short', 'Short', 'Medium', 'Long'] # 4 labels
    data[categories[i]] = pd.cut(
            data[featutes[i]],
            bins=cut_processed,
            labels=labels,
            include_lowest=True
        )


In [43]:
data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species,sepal_length_category,sepal_width_category,petal_length_category,petal_width_category
0,5.1,3.5,1.4,0.2,setosa,Very Short,Long,Very Short,Very Short
1,4.9,3.0,1.4,0.2,setosa,Very Short,Short,Very Short,Very Short
2,4.7,3.2,1.3,0.2,setosa,Very Short,Medium,Very Short,Very Short
3,4.6,3.1,1.5,0.2,setosa,Very Short,Medium,Very Short,Very Short
4,5.0,3.6,1.4,0.2,setosa,Very Short,Long,Very Short,Very Short
...,...,...,...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica,Long,Short,Long,Long
146,6.3,2.5,5.0,1.9,virginica,Long,Very Short,Medium,Long
147,6.5,3.0,5.2,2.0,virginica,Long,Short,Long,Long
148,6.2,3.4,5.4,2.3,virginica,Long,Long,Long,Long


In [47]:
columns_to_drop = ['sepal length (cm)','sepal width (cm)','petal length (cm)','petal width (cm)']
data = data.drop(columns=columns_to_drop)

In [48]:
data

,species,sepal_length_category,sepal_width_category,petal_length_category,petal_width_category
0,setosa,Very Short,Long,Very Short,Very Short
1,setosa,Very Short,Short,Very Short,Very Short
2,setosa,Very Short,Medium,Very Short,Very Short
3,setosa,Very Short,Medium,Very Short,Very Short
4,setosa,Very Short,Long,Very Short,Very Short
...,...,...,...,...,...
145,virginica,Long,Short,Long,Long
146,virginica,Long,Very Short,Medium,Long
147,virginica,Long,Short,Long,Long
148,virginica,Long,Long,Long,Long


In [49]:
species_column = data.pop("species")
data.insert(len(data.columns), "species", species_column)

In [50]:
data

,sepal_length_category,sepal_width_category,petal_length_category,petal_width_category,species
0,Very Short,Long,Very Short,Very Short,setosa
1,Very Short,Short,Very Short,Very Short,setosa
2,Very Short,Medium,Very Short,Very Short,setosa
3,Very Short,Medium,Very Short,Very Short,setosa
4,Very Short,Long,Very Short,Very Short,setosa
...,...,...,...,...,...
145,Long,Short,Long,Long,virginica
146,Long,Very Short,Medium,Long,virginica
147,Long,Short,Long,Long,virginica
148,Long,Long,Long,Long,virginica


In [51]:
train_data,test_data=train_test_split(data,test_size=0.2)

In [52]:
features = data.columns[:-1]
tree = id3(data, data, features, 'species')

In [53]:
pprint.pprint(tree)

{'petal_length_category': {'Long': 'virginica',
                           'Medium': {'petal_width_category': {'Long': {'sepal_width_category': {'Medium': {'sepal_length_category': {'Long': 'virginica',
                                                                                                                                      'Medium': 'versicolor'}},
                                                                                                 'Short': 'virginica',
                                                                                                 'Very Short': 'virginica'}},
                                                               'Medium': {'sepal_width_category': {'Medium': 'versicolor',
                                                                                                   'Short': 'versicolor',
                                                                                                   'Very Short': {'sepal_length_category': {'Long':

In [54]:
def predict(tree, instance):
    # If the current tree node is not a dictionary,
    # it is a leaf node → return its class label
    if not isinstance(tree, dict):
        return tree

    # Otherwise, extract the root feature used for splitting at this node
    root_node = next(iter(tree))

    # Get the instance's value for the root feature
    feature_value = instance[root_node]

    # If the value exists in the tree's branches, follow that branch recursively
    if feature_value in tree[root_node]:
        return predict(tree[root_node][feature_value], instance)

    # If the feature value was never seen during training, prediction is unknown
    return None


In [55]:
def evaluate(tree, test_data, label):
    correct_predict = 0
    for index,row in test_data.iterrows():
        result = predict(tree, row) 
        if result == row[label]:  
            correct_predict += 1

    accuracy = correct_predict / len(test_data)
    return accuracy

In [56]:
accuracy = evaluate(tree, test_data, 'species') 
print(accuracy)

0.9666666666666667
